# Week 9 live demo: data formats in health data science
Run top to bottom. Kernel: `week09` (see `README.md`). Each `##` section matches a segment in `W09L_script.md`.
The `!` lines run shell commands right in the notebook, so no second window is needed.

In [ ]:
%cd data
import os, time, json
import numpy as np, pandas as pd

## 1. Flat files: what goes wrong

In [ ]:
%%writefile flat_demo.tsv
sample_id	gene	count	collected
007	SEPTIN1	42	2026-03-02
008	MARCHF1	17	2026-03-02
009	TP53	5	2026-03-03

In [ ]:
df = pd.read_csv("flat_demo.tsv", sep="\t")
print(df.dtypes)
df

Look at `sample_id`: `007` became `7`. Dates are plain strings. Now tell pandas what we mean:

In [ ]:
df = pd.read_csv("flat_demo.tsv", sep="\t", dtype={"sample_id": str}, parse_dates=["collected"])
print(df.dtypes)
df

## 2. Columnar: Parquet (same table, three formats)

In [ ]:
rng = np.random.default_rng(9)
N = 500_000
labs = pd.DataFrame({
    "patient_id": np.arange(N),
    "age": rng.integers(18, 95, N),
    "sex": rng.choice(["F", "M"], N),
    "site": rng.choice(["DC", "MD", "VA", "NY"], N),
    "sbp": rng.normal(125, 15, N).round(1),
    "dbp": rng.normal(80, 10, N).round(1),
    "ldl": rng.normal(110, 30, N).round(1),
    "hba1c": rng.normal(5.9, 0.9, N).round(2),
    "visit_date": pd.to_datetime("2020-01-01") + pd.to_timedelta(rng.integers(0, 1500, N), unit="D"),
})
labs.head()

In [ ]:
import h5py
labs.to_csv("labs.csv", index=False)
labs.to_parquet("labs.parquet", index=False)
with h5py.File("labs.h5", "w") as f:
    for c in ["patient_id", "age", "sbp", "dbp", "ldl", "hba1c"]:
        f.create_dataset(c, data=labs[c].to_numpy(), compression="gzip")

def t(fn):
    s = time.perf_counter(); fn(); return time.perf_counter() - s

rows = [
    ("CSV",     "labs.csv",     t(lambda: pd.read_csv("labs.csv", usecols=["ldl"]))),
    ("Parquet", "labs.parquet", t(lambda: pd.read_parquet("labs.parquet", columns=["ldl"]))),
    ("HDF5",    "labs.h5",      t(lambda: h5py.File("labs.h5")["ldl"][:])),
]
pd.DataFrame([(n, os.path.getsize(p)/1e6, s) for n, p, s in rows],
             columns=["format", "size_MB", "read_one_column_s"]).round(3)

In [ ]:
import pyarrow.parquet as pq
print(pq.read_schema("labs.parquet"))          # the schema travels WITH the file
print(pq.ParquetFile("labs.parquet").metadata)   # row groups, per-column stats

Bridge from Week 8: SQL straight on the file, no import step.

In [ ]:
import duckdb
duckdb.sql("""
    SELECT site, count(*) AS n, round(avg(ldl), 1) AS mean_ldl
    FROM 'labs.parquet'
    WHERE age >= 65
    GROUP BY site ORDER BY site
""").df()

## 3. Hierarchical: HDF5 (a file system inside a file)

In [ ]:
rng = np.random.default_rng(1)
with h5py.File("toy_cells.h5", "w") as f:
    f.create_dataset("X", data=rng.poisson(2, (1000, 200)).astype("int32"), compression="gzip")  # cells x genes
    f.create_dataset("obs/cell_id", data=np.array([f"cell{i}" for i in range(1000)], dtype="S"))
    f.create_dataset("var/gene", data=np.array([f"gene{i}" for i in range(200)], dtype="S"))
    f["X"].attrs["description"] = "raw counts, cells x genes"

with h5py.File("toy_cells.h5") as f:
    f.visititems(lambda name, obj: print(name, obj))
    print(f["X"].attrs["description"])
    print(f["X"][:3, :5])          # read a slice; the other 99% never touches memory

## 4. FASTQ

In [ ]:
!head -n 8 reads.fastq

In [ ]:
!echo 'lines:' $(wc -l < reads.fastq) ' reads:' $(( $(wc -l < reads.fastq) / 4 ))

In [ ]:
import pysam
import matplotlib.pyplot as plt
quals = np.array([r.get_quality_array() for r in pysam.FastxFile("reads.fastq")])
print(quals.shape, "(reads x positions)")
plt.figure(figsize=(6, 3))
plt.plot(quals.mean(axis=0))
plt.axhline(30, ls="--", c="gray")
plt.xlabel("position in read"); plt.ylabel("mean Phred"); plt.title("Quality drops along the read")
plt.tight_layout(); plt.show()

## 5. Intervals: BED vs GFF3 vs GTF

In [ ]:
!cat genes.bed

In [ ]:
!grep -v '^#' genes.gff3 | head -6

In [ ]:
!head -n 3 genes.gtf

Same gene, two files. Compute its length from each:

In [ ]:
gff = pd.read_csv("genes.gff3", sep="\t", comment="#", header=None,
                  names="seqid source type start end score strand phase attributes".split())
bed = pd.read_csv("genes.bed", sep="\t", header=None, names="chrom start end name score strand".split())
a = gff[(gff.type == "gene") & gff.attributes.str.contains("Name=geneA")].iloc[0]
b = bed[bed.name == "geneA"].iloc[0]
print(f"GFF3 geneA: {a.start}-{a.end}  length = end - start + 1 = {a.end - a.start + 1}")
print(f"BED  geneA: {b.start}-{b.end}  length = end - start     = {b.end - b.start}")

## 6. SAM/BAM

In [ ]:
!samtools view -H reads.bam

In [ ]:
!samtools view reads.bam | head -n 3 | cut -f1-11

In [ ]:
!samtools view reads.bam chrT1:495-505 | cut -f1-6

In [ ]:
!samtools depth -r chrT1:498-502 reads.bam

Coordinate trap #2: samtools shows 1-based `POS`; pysam's `reference_start` is 0-based.

In [ ]:
bam = pysam.AlignmentFile("reads.bam")
first = next(bam.fetch())
print("pysam reference_start (0-based):", first.reference_start)
print("SAM POS (1-based):              ", first.reference_start + 1)

In [ ]:
from collections import Counter
for col in bam.pileup("chrT1", 499, 500, truncate=True, min_base_quality=0):
    print(f"chrT1:{col.reference_pos + 1}  depth={col.n}")
    print(Counter(b.upper() for b in col.get_query_sequences()))

## 7. VCF

In [ ]:
!grep -v '^##' variants.vcf

In [ ]:
!bcftools view -H -r chrT1:490-510 variants.vcf.gz

In [ ]:
vcf = pysam.VariantFile("variants.vcf.gz")
rows = [(r.chrom, r.pos, r.ref, r.alts[0], r.qual, list(r.filter)[0], r.info["DP"], r.info["AF"][0],
         r.samples["SAMPLE1"]["GT"]) for r in vcf]
variants = pd.DataFrame(rows, columns=["chrom", "pos", "ref", "alt", "qual", "filter", "dp", "af", "gt"])
variants

Close the loop: reads (BAM) show the allele, the VCF calls it, the GFF says where it lives.

In [ ]:
exons = gff[gff.type == "exon"]
for _, v in variants[variants["filter"] == "PASS"].iterrows():
    hit = exons[(exons.seqid == v.chrom) & (exons.start <= v.pos) & (v.pos <= exons.end)]
    print(f"{v.chrom}:{v.pos} {v.ref}>{v.alt}  ->", hit.attributes.str.extract(r"ID=([^;]+)")[0].tolist() or "intergenic/intronic")

## 8. JSON and APIs

In [ ]:
import requests
URL = "https://rest.ensembl.org/lookup/symbol/homo_sapiens/KRAS"
try:
    r = requests.get(URL, headers={"Content-Type": "application/json"}, timeout=10)
    print("status:", r.status_code)
    r.raise_for_status()
    gene = r.json()
except Exception as e:                       # classroom Wi-Fi happens
    print("live call failed, using saved copy:", type(e).__name__)
    gene = json.load(open("ensembl_kras_saved.json"))
{k: gene[k] for k in ["display_name", "id", "seq_region_name", "start", "end", "strand", "biotype", "assembly_name"]}

Nested JSON to a table: a synthetic FHIR-style patient bundle.

In [ ]:
bundle = json.load(open("synthetic_fhir_patients.json"))
pd.json_normalize(bundle["entry"], sep="_")

In [ ]:
pts = pd.json_normalize(bundle["entry"], record_path=["resource", "name"],
                        meta=[["resource", "id"], ["resource", "gender"]])
pts["given"] = pts["given"].str[0]       # lists inside cells: flatten on purpose
pts